# Лабораторная работа 3: Машины опорных векторов (SVM) для детекции P300

**Задание:**
- Использовать набор данных из `moabb`.
- Проанализировать данные и обосновать способ детекции признаков.
- Обучить SVM-модель для определения потенциала P300.
- Посчитать `accuracy`, `precision`, `recall`, `F1`.
- Построить ROC-кривую.

> По уточнению преподавателя в работе можно использовать подготовленный датасет через `moabb.paradigms.P300`.

In [ ]:
# Если библиотек нет, раскомментируйте установку:
# %pip install -U moabb mne scikit-learn pandas numpy matplotlib seaborn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from moabb.datasets import bi2013a
from moabb.paradigms import P300

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_curve,
    auc,
    confusion_matrix,
    classification_report,
)

sns.set(style="whitegrid")
np.random.seed(42)

## 1) Загрузка и первичный анализ данных MOABB

Для воспроизводимости берём классический P300-датасет `bi2013a` и получаем уже подготовленные эпохи через `P300()`.

Ожидаемая форма массива:
- `X`: `(n_epochs, n_channels, n_times)`
- `y`: метки классов (`Target` / `NonTarget`)
- `metadata`: служебная информация (субъект, сессия и т.д.)

In [ ]:
dataset = bi2013a.BI2013a()
paradigm = P300()

X, y, metadata = paradigm.get_data(dataset=dataset)


print("X shape:", X.shape)
print("y shape:", y.shape)
print("Classes:", pd.Series(y).value_counts().to_dict())
metadata.head()

## 2) Детекция и обоснование признаков

Для P300 важен поздний положительный компонент примерно в окне **250-500 мс** после стимула.

Поэтому используем признаки, которые напрямую описывают форму ERP в этом диапазоне:
1. **Средняя амплитуда в окне 250-500 мс** по каждому каналу.
2. **Максимальная амплитуда в окне 250-500 мс** по каждому каналу.
3. **Латентность пика** (индекс времени максимума в окне) по каждому каналу.

Такие признаки физиологически интерпретируемы и компактнее, чем использование всех отсчётов целиком.

In [ ]:
# Восстанавливаем временную ось эпохи
n_times = X.shape[2]
tmin = float(paradigm.tmin)
tmax = float(paradigm.tmax)

# Если resample задан, берём его; иначе оцениваем по длительности окна
if paradigm.resample is not None:
    sfreq = float(paradigm.resample)
else:
    sfreq = (n_times - 1) / (tmax - tmin)

times = np.linspace(tmin, tmax, n_times)

# Окно P300: 250-500 мс
mask = (times >= 0.25) & (times <= 0.50)

X_win = X[:, :, mask]

# Признаки
mean_amp = X_win.mean(axis=2)
max_amp = X_win.max(axis=2)

# Латентность считаем относительно начала эпохи
peak_idx = X_win.argmax(axis=2)
peak_latency = peak_idx / sfreq + 0.25

X_features = np.concatenate([mean_amp, max_amp, peak_latency], axis=1)

print("Feature matrix shape:", X_features.shape)  # (n_epochs, n_channels * 3)

## 3) Обучение SVM и обоснование выбора

Выбираем **C-SVC с RBF-ядром**:
- `SVC` (классический C-Support Vector Classification) подходит для бинарной задачи `Target/NonTarget`.
- `RBF` полезен для нелинейно разделимых EEG-признаков.
- Перед моделью используем стандартизацию, так как SVM чувствителен к масштабу признаков.

Гиперпараметры:
- `C=2.0` — умеренный баланс между штрафом за ошибки и обобщением.
- `gamma='scale'` — стандартная устойчивая настройка для RBF.
- `class_weight='balanced'` — учитываем дисбаланс классов (обычно `Target` встречается реже).

In [ ]:
le = LabelEncoder()
y_bin = le.fit_transform(y)  # Target/NonTarget -> 0/1

X_train, X_test, y_train, y_test = train_test_split(
    X_features,
    y_bin,
    test_size=0.25,
    random_state=42,
    stratify=y_bin,
)

svm_clf = Pipeline([
    ("scaler", StandardScaler()),
    (
        "svc",
        SVC(
            kernel="rbf",
            C=2.0,
            gamma="scale",
            probability=True,
            class_weight="balanced",
            random_state=42,
        ),
    ),
])

svm_clf.fit(X_train, y_train)

y_pred = svm_clf.predict(X_test)
y_proba = svm_clf.predict_proba(X_test)[:, 1]  # вероятность класса 1

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-score : {f1:.4f}")

print("\nClassification report:\n")
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
# Матрица ошибок
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title('Confusion Matrix (SVM, P300)')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.show()

In [ ]:
# ROC-кривая
fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, lw=2, label=f'ROC curve (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve for P300 Detection (SVM)')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 4) Вывод

В работе:
- использован подготовленный P300-датасет из `moabb.paradigms.P300`;
- предложены интерпретируемые ERP-признаки в окне 250-500 мс;
- обучена SVM-модель с RBF-ядром;
- рассчитаны `accuracy`, `precision`, `recall`, `F1` и построена ROC-кривая.

При желании для улучшения качества можно добавить кросс-валидацию по субъектам и подбор гиперпараметров (`GridSearchCV`).